In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from datetime import timedelta

##  Analyse time series data of WEather uNDERground

In [2]:
df = pd.read_parquet('/home/camarada/Documents/projects/temp-grss-nasa/data_/wunderground/Revision_temp_time.parquet')
df['time_max'] = pd.to_datetime(df['time_of_max_temp'], format= "%I:%M %p",errors='coerce').dt.time  
df['time_min'] = pd.to_datetime(df['time_of_min_temp'], format= "%I:%M %p",errors='coerce').dt.time 

## get date 
df['date_str'] = df['source_file'].apply(lambda x:x.strip().split('_',1)[1].replace('_','-'))
df['date']= pd.to_datetime(df['date_str'], format='%Y-%m-%d')
df['datetime_max']= pd.to_datetime(df['date_str'] + ' ' + df['time_of_max_temp'], format='%Y-%m-%d %I:%M %p')
df['datetime_min']= pd.to_datetime(df['date_str'] + ' ' + df['time_of_min_temp'], format='%Y-%m-%d %I:%M %p')

## diference between max min temperature 
df['diff_temp'] = df['max_temp'] - df['min_temp']

# difference of time between the min and the max 
## pass to time delta to calculate the diff
df['diff_time'] = df['time_max'].apply(lambda x: timedelta(hours=x.hour, minutes=x.minute)) - df['time_min'].apply(lambda x: timedelta(hours=x.hour, minutes=x.minute))

## add season column
df['season'] = df['date'].dt.month%12 // 3 + 1
df['season'] = df['season'].map({1: 'Winter', 2: 'Spring', 3: 'Summer', 4: 'Fall'})

In [52]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2202 entries, 0 to 2201
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype          
---  ------            --------------  -----          
 0   source_file       2202 non-null   object         
 1   max_temp          2202 non-null   float64        
 2   min_temp          2202 non-null   float64        
 3   time_of_max_temp  2202 non-null   object         
 4   time_of_min_temp  2202 non-null   object         
 5   time_max          2202 non-null   object         
 6   time_min          2202 non-null   object         
 7   date_str          2202 non-null   object         
 8   date              2202 non-null   datetime64[ns] 
 9   datetime_max      2202 non-null   datetime64[ns] 
 10  datetime_min      2202 non-null   datetime64[ns] 
 11  diff_temp         2202 non-null   float64        
 12  diff_time         2202 non-null   timedelta64[ns]
 13  season            2202 non-null   object         
dtypes: datet

In [4]:
# 1. Define your threshold
threshold_str = "13:20"
h, m = map(int, threshold_str.split(':'))
threshold_minutes = h * 60 + m

# 2. Filter the DataFrame
# We calculate (hour * 60 + minute) for every record
filtered_df = df[
    (df['datetime_max'].dt.hour * 60 + df['datetime_max'].dt.minute) > threshold_minutes
].copy()

# 3. Find the month with the highest temperature after this time
if not filtered_df.empty:
    hottest_row = filtered_df.loc[filtered_df['max_temp'].idxmax()]
    hottest_month = hottest_row['date'].strftime('%B')
    print(f"The highest temperature after {threshold_str} was {hottest_row['max_temp']}°C in {hottest_month}.")



The highest temperature after 13:20 was 37.22222222222222°C in July.


In [ ]:
from bokeh.plotting import figure, show, output_file
from bokeh.models import ColumnDataSource, HoverTool, Span
from bokeh.layouts import column

# Prepare the data
# (Assuming 'df' is your loaded DataFrame)
df['time_str'] = df['datetime_max'].dt.strftime('%H:%M')
df['date_label'] = df['date'].dt.strftime('%Y-%m-%d')

# Filter for the plot (static example for 13:20)
threshold_time = 13 * 60 + 20
df['is_after_threshold'] = (df['datetime_max'].dt.hour * 60 + df['datetime_max'].dt.minute) > threshold_time

# Create DataSources
source_all = ColumnDataSource(df[~df['is_after_threshold']])
source_filtered = ColumnDataSource(df[df['is_after_threshold']])

# Create Figure
p = figure(title=f"Max Temps After 13:20", 
           x_axis_type="datetime", 
           height=500, width=900,
           x_axis_label='Date', y_axis_label='Max Temperature (°C)')

# Plot background data (before 13:20)
p.scatter('date', 'max_temp', source=source_all, 
          color='lightgray', size=6, alpha=0.4, legend_label="Before 13:20")

# Plot highlighted data (after 13:20)
p.scatter('date', 'max_temp', source=source_filtered, 
          color='#D95F02', size=10, alpha=0.8, legend_label="After 13:20")

# Add Hover Tool
hover = HoverTool(tooltips=[
    ("Date", "@date_label"),
    ("Time of Max", "@time_str"),
    ("Max Temp", "@max_temp{0.0}°C"),
    ("Season", "@season")
])
p.add_tools(hover)

# Legend formatting
p.legend.click_policy = "hide"
p.legend.location = "top_left"

output_notebook()
show(p)


In [3]:
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.palettes import Category10

output_notebook()

df = df.sort_values("date")

## convert to celsius
df['max_temp'] = (df['max_temp'] - 32) * 5.0/9.0
df['min_temp'] = (df['min_temp'] - 32) * 5.0/9.0
source = ColumnDataSource(df)

p = figure(
    x_axis_type="datetime",
    width=900,
    height=400,
    title="Daily Max and Min Temperature",
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

p.line('date', 'max_temp', source=source,
       color=Category10[3][0], legend_label="Max Temp", line_width=2)

p.line('date', 'min_temp', source=source,
       color=Category10[3][1], legend_label="Min Temp", line_width=2)

hover = HoverTool(
    tooltips=[
        ("Date", "@date{%F}"),
        ("Max Temp", "@max_temp"),
        ("Min Temp", "@min_temp")
    ],
    formatters={'@date': 'datetime'},
    mode='vline'
)

p.add_tools(hover)

p.legend.click_policy = "hide"
p.xaxis.axis_label = "Date"
p.yaxis.axis_label = "Temperature"

show(p)

Loading BokehJS ...

## Compare data providers

- Uk Meteo 2km with in situ. from Weather Underground

In [1]:
import pandas as pd
import numpy as np 
import altair as alt
from pathlib import Path
from datetime import date
import polars as pl


In [2]:
yearr = '2024'
uk_meteo_path = f'/home/camarada/Documents/projects/temp-grss-nasa/data_/UKMet2km_london_hourly/londocityairport_UKV_{yearr}.parquet'
df = pd.read_parquet(uk_meteo_path)

## rename to datetime to match
df.rename(columns={'date':"datetime"}, inplace= True)

## needs to be utc aware, in case analysing another point
# 1. Ensure it is UTC aware (Open-Meteo usually is)
# 2. Convert to London Local Time (handles GMT/BST automatically)
# 3. Strip the zone info so it matches your other naive datasets
df['datetime'] = (df['datetime']
                  .dt.tz_convert('Europe/London')
                  .dt.tz_localize(None))

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8784 entries, 0 to 8783
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   datetime                 8784 non-null   datetime64[ns]
 1   temperature_2m           8784 non-null   float32       
 2   cloud_cover              8784 non-null   float32       
 3   cloud_cover_low          8784 non-null   float32       
 4   cloud_cover_mid          8784 non-null   float32       
 5   cloud_cover_high         8784 non-null   float32       
 6   dew_point_2m             8784 non-null   float32       
 7   pressure_msl             8784 non-null   float32       
 8   surface_pressure         8784 non-null   float32       
 9   wind_speed_10m           8784 non-null   float32       
 10  vapour_pressure_deficit  8784 non-null   float32       
 11  shortwave_radiation      8784 non-null   float32       
 12  direct_radiation         8784 non-

In [2]:
RAW_DIR = Path(f"/home/camarada/Documents/projects/temp-grss-nasa/data_/wunderground/processed/{yearr}")
year = sorted(RAW_DIR.rglob("*.parquet"))

# 1. Load all years at once — Polars handles this efficiently
df_wupl = pl.scan_parquet(year).select(
                (pl.col("Temperature").sub(32).mul((5/9)).alias("WU_temp"), ## convert to celcius
                pl.col("datetime"))
                ).collect()

df_wupl = df_wupl.to_pandas()

NameError: name 'yearr' is not defined

In [4]:
## dealing with clock change in summer periods
## when the clocks goes back from summer, it may cause
## duplicated times period which causes error when concatenating 

# Group by hour, mean it, and then drop any index duplicates that remain
wu_resampled = df_wupl.set_index('datetime').resample('1h').mean()
wu_resampled = wu_resampled[~wu_resampled.index.duplicated(keep='first')]

## do the same for openmeteo UK2KM 
df_uk = df.set_index('datetime').resample('1h').mean()
df_uk = df_uk[~df_uk.index.duplicated(keep='first')]

# Now the concat will work
dff = pd.concat([
    df_uk[['temperature_2m']],
    wu_resampled
],
    axis=1)

## modify name
dff = dff.rename(columns={
            'temperature_2m':'UK2km',
            'WU_temp':'WeaUnder'})
dff.head()

,UK2km,WeaUnder
datetime,,
2024-01-01 00:00:00,8.9305,8.888889
2024-01-01 01:00:00,8.7805,8.333333
2024-01-01 02:00:00,8.6805,7.777778
2024-01-01 03:00:00,8.6805,8.888889
2024-01-01 04:00:00,8.8805,8.333333


In [28]:
df_wupl[df_wupl['datetime'].duplicated()]

,WU_temp,datetime
14347,10.0,2024-10-27 02:20:00
14348,10.0,2024-10-27 02:50:00


In [5]:
season_map = {
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "autumn", 10: "autumn", 11: "autumn"
}

dff["season"] = dff.reset_index()["datetime"].dt.month.map(season_map).values

dff['month'] = dff.reset_index()["datetime"].dt.month_name()

In [17]:
## groupby day retrieving the max temp
daily_agg = dff[['UK2km','WeaUnder']].groupby(
                        pd.Grouper(freq='1d')
).max()

daily_agg['error'] = daily_agg['WeaUnder'] - daily_agg['UK2km']

daily_agg['sq_error'] = daily_agg['error']**2
daily_agg['month'] = daily_agg.reset_index()['datetime'].dt.month_name().values

In [24]:
month_order = [
    'January','February','March','April','May','June',
    'July','August','September','October','November','December'
]

error_bars = alt.Chart(
    daily_agg.reset_index()[['datetime','error','month']]
).mark_errorbar(
        extent = 'stdev'
).encode(
    x =alt.X('error:Q', title='Error'),
    y = alt.Y('month:N',title='2024', sort=month_order)
)

points = alt.Chart(
    daily_agg.reset_index()[['datetime','error','month']]
).mark_point(
    filled=True,
    color='black'
).encode(
    x=alt.X('mean(error)'),
    y=alt.Y('month:N', sort=month_order)
)
chart = (error_bars + points).properties(
    title='Monthly Error - UK Met2Km & WeatherUnder',
    width=500,
    height=350
)
chart

alt.LayerChart(...)

In [ ]:
month_order = [
    'Jan','Feb','Mar','Apr','May','Jun',
    'Jul','Aug','Sep','Oct','Nov','Dec'
]

base = alt.Chart(
    daily_agg.reset_index()[['datetime','error','month']]
).transform_density(
    'error',
    groupby=['month'],
    as_=['error', 'density']
)

ridge = base.mark_area(
    interpolate='monotone',
    fillOpacity=0.7,
    stroke='black'
).encode(
    x=alt.X('error:Q', title='Error'),
    y=alt.Y(
        'density:Q',
        stack='center',
        axis=None
    ),
    row=alt.Row(
        'month:N',
        sort=month_order,
        title=None
    )
).properties(
    width=500,
    height=60,
    title='Distribution of Error by Month'
)

zero = alt.Chart(pd.DataFrame({'x':[0]})).mark_rule(
    color='red'
).encode(x='x:Q')

ridge + zero

alt.Chart(...)

### PLot for the whole historical data available
2022-2025

### Create Files for Reporting

In [3]:
RAW_DIR = Path("/home/camarada/Documents/projects/temp-grss-nasa/data_/wunderground/processed")

files = sorted(RAW_DIR.rglob("*.parquet"))

df_wupl = (
    pl.scan_parquet(files)
    .select(
        (pl.col("Temperature").sub(32).mul(5/9)).alias("WU_temp"),
        pl.col("datetime")
    )
    .filter(
        pl.col("datetime").dt.year().is_between(2022, 2025)
    )
    .collect()
)

df_wupl = df_wupl.to_pandas()

In [4]:
df_wupl.shape

(67885, 2)

In [5]:
years = range(2022, 2026)

dfs = []

for y in years:
    path = f'/home/camarada/Documents/projects/temp-grss-nasa/data_/UKMet2km_london_hourly/londocityairport_UKV_{y}.parquet'
    tmp = pd.read_parquet(path)

    tmp.rename(columns={'date': 'datetime'}, inplace=True)

    tmp['datetime'] = (
        tmp['datetime']
        .dt.tz_convert('Europe/London')
        .dt.tz_localize(None)
    )

    dfs.append(tmp)

df = pd.concat(dfs, ignore_index=True)

In [6]:
df_wupl.shape, df.shape

((67885, 2), (33648, 14))

In [7]:
print(df['datetime'].min(), df['datetime'].max())
print(df_wupl['datetime'].min(), df_wupl['datetime'].max())

2022-03-01 00:00:00 2025-12-31 23:00:00
2022-01-01 23:20:00 2025-12-01 23:50:00


In [8]:
max(df['datetime'].min(),df_wupl['datetime'].min() )

Timestamp('2022-03-01 00:00:00')

In [9]:
## filter by min and max date
## first by min date between both datasets, take the max
df_filt =  df.loc[df['datetime'] >= max(df['datetime'].min(),df_wupl['datetime'].min() )]
df_wupl_filt =  df_wupl.loc[df_wupl['datetime'] >= max(df['datetime'].min(),df_wupl['datetime'].min() )]

## slice 
df_filt =  df_filt.loc[df_filt['datetime'] < min(df['datetime'].max(),df_wupl['datetime'].max() )]
df_wupl_filt =  df_wupl_filt.loc[df_wupl_filt['datetime'] < min(df['datetime'].max(),df_wupl['datetime'].max() )]

## dealing with clock change in summer periods
## when the clocks goes back from summer, it may cause
## duplicated times period which causes error when concatenating 

# Group by hour, mean it, and then drop any index duplicates that remain
wu_resampled = df_wupl_filt.set_index('datetime').resample('1h').mean()
wu_resampled = wu_resampled[~wu_resampled.index.duplicated(keep='first')]

## do the same for openmeteo UK2KM 
df_uk = df_filt.set_index('datetime').resample('1h').mean()
df_uk = df_uk[~df_uk.index.duplicated(keep='first')]

# Now the concat will work
dff = pd.concat([
    df_uk[['temperature_2m']],
    wu_resampled
],
    axis=1)

## modify name
dff = dff.rename(columns={
            'temperature_2m':'UK2km',
            'WU_temp':'WeaUnder'})
dff.head()

,UK2km,WeaUnder
datetime,,
2022-03-01 00:00:00,9.880500,10.000000
2022-03-01 01:00:00,9.830501,8.888889
2022-03-01 02:00:00,9.830501,8.888889
2022-03-01 03:00:00,9.780500,9.444444
2022-03-01 04:00:00,9.780500,8.888889


In [30]:
import os 
folder = "/home/camarada/Documents/projects/temp-grss-nasa/daily_max_temp/explore_temp_and_time_distribution/historic_uk2m_aggregated"
daily_agg.to_csv(os.path.join(folder,"daily_agg_2022-2025_uk2m"))

In [ ]:
day_stats.loc['']

,mean,std,max
month_day,,,
01-01,0.536167,0.598326,1.191722
01-02,-0.087907,0.478853,0.408389
01-03,0.052833,0.513371,0.591722
01-04,0.112092,0.439428,0.619500
01-05,0.084315,0.064150,0.158389
...,...,...,...
12-27,0.036166,0.100462,0.141722
12-28,0.256537,0.260658,0.541722
12-29,0.239870,0.494611,0.741722


In [16]:
import datetime

In [27]:
analysis_day = "2022-03-12"
#datetime.datetime.strptime(analysis_day,format="%y-%m-%d")

for i, row in day_stats.loc['03-12'].items():
    print(f"{i} : {row:.2f}")


mean : 0.28
std : 0.32
max : 0.66


In [14]:
df = daily_agg.copy()

# --- 1. Aggregate by day of year (e.g. March 12 = "03-12") ---
df['month_day'] = df.index.strftime('%m-%d')

day_stats = df.groupby('month_day')['error'].agg(['mean', 'std', 'max',])
print(day_stats.loc['03-12'])  # swap for any day


# # --- 2. First week of December ---
# dec_week1 = df[(df.index.month == 12) & (df.index.day <= 7)]

# dec_week1_stats = dec_week1['error'].agg(['mean', 'std', 'max'])
# print(dec_week1_stats)


# # --- 3. Monthly aggregation ---
# monthly_stats = df.groupby('month')['error'].agg(['mean', 'std', 'max'])
# print(monthly_stats)

# # If you want months in calendar order rather than alphabetical:
# month_order = ['January','February','March','April','May','June',
#                'July','August','September','October','November','December']
# monthly_stats = monthly_stats.reindex(month_order)


# # --- 4. Total (overall) error mean and std ---
# total_stats = df['error'].agg(['mean', 'std'])
# print(total_stats)

mean    0.283389
std     0.319996
max     0.658389
Name: 03-12, dtype: float64


In [10]:
## groupby day retrieving the max temp
daily_agg = dff[['UK2km','WeaUnder']].groupby(
                        pd.Grouper(freq='1d')
).max()

daily_agg['error'] = daily_agg['WeaUnder'] - daily_agg['UK2km']

daily_agg['sq_error'] = daily_agg['error']**2
daily_agg['month'] = daily_agg.reset_index()['datetime'].dt.month_name().values
daily_agg['year'] = daily_agg.reset_index()['datetime'].dt.year.values


In [56]:
boxplot = alt.Chart(
    daily_agg,
    width=400
).mark_boxplot(extent='min-max').encode(
    x='year:O',
    y='error:Q'
)

chart = (boxplot).properties(
    title='Error of UKmeteo and WeUnder per year',
    width=500,
    height=350
)
chart

alt.Chart(...)

In [58]:

month_order = [
    'January','February','March','April','May','June',
    'July','August','September','October','November','December'
]

base = alt.Chart(daily_agg.reset_index())

error_bars = base.mark_errorbar(
    extent='ci'
).encode(
    x=alt.X('error:Q', title='Error'),
    y=alt.Y('month:N', sort=month_order, title='Month')
)

points = base.mark_point(
    filled=True,
    color='black',
    size=60
).encode(
    x=alt.X('mean(error):Q'),
    y=alt.Y('month:N', sort=month_order)
)

chart = (error_bars + points).properties(
    title='Mean Error with 95% Confidence Interval by Month',
    width=500,
    height=350
)

chart

alt.LayerChart(...)

In [61]:
from scipy import stats

def monthly_stats(x):
    
    n = len(x)
    mean = np.mean(x)
    std = np.std(x, ddof=1)

    ci_low, ci_high = stats.t.interval(
        confidence=0.95,
        df=n-1,
        loc=mean,
        scale=stats.sem(x)
    )

    return pd.Series({
        "n": n,
        "mean_error": mean,
        "std": std,
        "ci_low": ci_low,
        "ci_high": ci_high
    })

monthly_summary = daily_agg.groupby("month")["error"].apply(monthly_stats)

In [62]:
monthly_summary

month                
April      n             120.000000
           mean_error     -0.333972
           std             0.506863
           ci_low         -0.425591
           ci_high        -0.242353
August     n             124.000000
           mean_error     -0.051153
           std             0.486763
           ci_low         -0.137680
           ci_high         0.035373
December   n              94.000000
           mean_error      0.094795
           std             0.463314
           ci_low         -0.000101
           ci_high         0.189691
February   n              85.000000
           mean_error     -0.090892
           std             0.412057
           ci_low         -0.179771
           ci_high        -0.002014
January    n              93.000000
           mean_error      0.011794
           std             0.409662
           ci_low         -0.072575
           ci_high         0.096163
July       n             124.000000
           mean_error     -0.123286
      

In [72]:
daily_agg.head()

,UK2km,WeaUnder,error,sq_error,month,year
datetime,,,,,,
2022-03-01,9.8805,10.000000,0.119500,0.014280,March,2022
2022-03-02,8.1305,7.777778,-0.352722,0.124413,March,2022
2022-03-03,12.9805,12.777778,-0.202722,0.041096,March,2022
2022-03-04,9.7305,10.000000,0.269500,0.072630,March,2022
2022-03-05,7.3305,7.500000,0.169500,0.028730,March,2022


In [73]:

season_map = {
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "autumn", 10: "autumn", 11: "autumn"
}

daily_agg["season"] = daily_agg.reset_index()["datetime"].dt.month.map(season_map).values
source = (
        (daily_agg.reset_index().drop(
            columns=['UK2km','WeaUnder']
            ).melt(id_vars=['datetime','season'],
                value_vars=['error']   
                )[["datetime", "season", "value"]])
        .reset_index()
       .set_index("datetime")
       .groupby("season")
       .resample("1D")["value"]
       .mean()
       .reset_index()
)
step = 80
overlap = 0.5

chart = (
    alt.Chart(source, height=step, width=600)
    .transform_bin(
        ["bin_max", "bin_min"], "value"
    )
    .transform_aggregate(
        value="count()",
        groupby=["season", "bin_min", "bin_max"]
    )
    .transform_impute(
        impute="value",
        groupby=["season"],
        key="bin_min",
        value=0
    )
    .mark_area(
        interpolate="monotone",
        fillOpacity=0.8,
        stroke="lightgray",
        strokeWidth=0.5,
        clip=True
    )
    .encode(
        alt.X("bin_min:Q", title="Temperature Difference"),
        alt.Y(
            "value:Q",
            axis=None,
            scale=alt.Scale(range=[step, -step * overlap])
        ),
        alt.Color("season:N")
    )
    .facet(
        row=alt.Row(
            "season:N",
            header=alt.Header(labelAngle=0, labelAlign="left")
        )
    )
    .configure_facet(spacing=0)
    .configure_view(stroke=None)
)

chart

alt.FacetChart(...)

In [69]:
dff

,UK2km,WeaUnder,season
datetime,,,
2022-03-01 00:00:00,9.880500,10.000000,spring
2022-03-01 01:00:00,9.830501,8.888889,spring
2022-03-01 02:00:00,9.830501,8.888889,spring
2022-03-01 03:00:00,9.780500,9.444444,spring
2022-03-01 04:00:00,9.780500,8.888889,spring
...,...,...,...
2025-12-01 19:00:00,11.330501,11.111111,winter
2025-12-01 20:00:00,10.830501,11.111111,winter
2025-12-01 21:00:00,10.180500,11.111111,winter
